# Amazon Reviews - Cleaned Data Explorer & Editor - sample of 25 records for each of 8 files

This notebook allows you to:
1. Load and inspect the cleaned dataset
2. View summary statistics
3. Filter and explore records
4. Make edits as needed
5. Export updated data

**Data Source**: `data/cleaned/cleaned_reviews.csv`

**Last Generated**: Check DATA_QUALITY_REPORT.md in docs/ for details

## Setup

In [ ]:
import pandas as pd
import numpy as np
from datetime import datetime
from pathlib import Path
import warnings
warnings.filterwarnings('ignore')

# Set pandas display options
pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', 100)
pd.set_option('display.width', None)

# Paths - relative to notebook location
PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
CLEANED_DATA_PATH = PROJECT_ROOT / 'data/cleaned/cleaned_reviews.csv'
DOCS_DIR = PROJECT_ROOT / 'docs'

print(f"Project Root: {PROJECT_ROOT}")
print(f"Data Path: {CLEANED_DATA_PATH}")
print(f"File exists: {CLEANED_DATA_PATH.exists()}")

## Load Data

In [2]:
# Load cleaned data
df = pd.read_csv(CLEANED_DATA_PATH)

print(f"Loaded {len(df):,} records")
print(f"\nShape: {df.shape}")
print(f"\nColumns: {list(df.columns)}")
print(f"\nData types:")
print(df.dtypes)

Loaded 2,516,345 records

Shape: (2516345, 12)

Columns: ['user_id', 'parent_asin', 'asin', 'timestamp', 'date', 'rating', 'verified_purchase', 'category_name', 'helpful_vote', 'user_first_date', 'days_since_first', 'review_sequence']

Data types:
user_id               object
parent_asin           object
asin                  object
timestamp              int64
date                  object
rating               float64
verified_purchase       bool
category_name         object
helpful_vote           int64
user_first_date       object
days_since_first       int64
review_sequence        int64
dtype: object


## Quick Overview

In [3]:
# Display first few rows
print("First 5 records:")
df.head()

First 5 records:


,user_id,parent_asin,asin,timestamp,date,rating,verified_purchase,category_name,helpful_vote,user_first_date,days_since_first,review_sequence
0,AE222FP7YRNFCEQ2W3ZDIGMSYTLQ,B0BZPH4FBL,B0B76T5J93,1675445632132,2023-02-03 12:33:52.132,5.0,True,Cell_Phones_and_Accessories,0,2023-02-03 12:33:52.132,0,1
1,AE222GHCPAOQIDZDVHIMMGZ6WZSA,B0B12FPWCH,B0B11Q9SFK,1682686414935,2023-04-28 08:53:34.935,5.0,True,Cell_Phones_and_Accessories,0,2023-04-28 08:53:34.935,0,1
2,AE222HGWAAOGMKHHY5HIIRBFEW2Q,B0B2SGLPT1,B098N7GRJV,1683230749825,2023-05-04 16:05:49.825,1.0,True,Cell_Phones_and_Accessories,2,2023-05-04 16:05:49.825,0,1
3,AE2236OE2DFF4DJNVOIXEQ5ARARQ,B0BT55QBDN,B0BT55QBDN,1683890066657,2023-05-12 07:14:26.657,5.0,True,Cell_Phones_and_Accessories,1,2023-05-12 07:14:26.657,0,1
4,AE2236YX2UYRICEGHLDLIGRWBGAQ,B09Y24QCL9,B09Y23QGDD,1676562569177,2023-02-16 10:49:29.177,5.0,True,Cell_Phones_and_Accessories,0,2023-02-16 10:49:29.177,0,1


In [4]:
# Summary statistics
print("\n=== SUMMARY STATISTICS ===")
print(f"Total records: {len(df):,}")
print(f"\nCategories: {df['category_name'].nunique()}")
print(f"Breakdown:")
print(df['category_name'].value_counts())

print(f"\nUnique users: {df['user_id'].nunique():,}")
print(f"Unique products (parent_asin): {df['parent_asin'].nunique():,}")
print(f"\nDate range: {df['date'].min()} to {df['date'].max()}")

print(f"\nverified_purchase values:")
print(df['verified_purchase'].value_counts())


=== SUMMARY STATISTICS ===
Total records: 2,516,345

Categories: 4
Breakdown:
category_name
Electronics                    1535698
Cell_Phones_and_Accessories     810361
Video_Games                     142623
Software                         27663
Name: count, dtype: int64

Unique users: 1,827,354
Unique products (parent_asin): 369,157

Date range: 2023-01-01 00:00:02.696 to 2023-06-29 23:59:46.366

verified_purchase values:
verified_purchase
True    2516345
Name: count, dtype: int64


In [5]:
# Null values check
print("\n=== NULL VALUES ===")
null_counts = df.isnull().sum()
if null_counts.sum() > 0:
    print(null_counts[null_counts > 0])
else:
    print("✓ No null values detected")


=== NULL VALUES ===
✓ No null values detected


## Retention Analysis Metrics

In [6]:
# Check retention column derivation
print("=== RETENTION COLUMNS ===")
print(f"\nuser_first_date (sample):")
print(df[['user_id', 'date', 'user_first_date', 'days_since_first', 'review_sequence']].head(10))

print(f"\n\nReview sequence distribution:")
print(df['review_sequence'].value_counts().sort_index().head(10))

print(f"\n\nDays since first review distribution:")
print(df['days_since_first'].describe())

=== RETENTION COLUMNS ===

user_first_date (sample):
                        user_id                     date  \
0  AE222FP7YRNFCEQ2W3ZDIGMSYTLQ  2023-02-03 12:33:52.132   
1  AE222GHCPAOQIDZDVHIMMGZ6WZSA  2023-04-28 08:53:34.935   
2  AE222HGWAAOGMKHHY5HIIRBFEW2Q  2023-05-04 16:05:49.825   
3  AE2236OE2DFF4DJNVOIXEQ5ARARQ  2023-05-12 07:14:26.657   
4  AE2236YX2UYRICEGHLDLIGRWBGAQ  2023-02-16 10:49:29.177   
5  AE22372J7JGUBWMHLNNR4MTMLILA  2023-01-03 22:39:33.581   
6  AE223DG6TC7OPPAVKE6NGBOWNZAQ  2023-01-01 17:37:36.763   
7  AE223EZCLCLS6FE3LC7B7TTBR6KA  2023-02-16 12:42:58.510   
8  AE223FOBUQAUOSGVCDRABQGKGVSQ  2023-01-14 14:10:30.367   
9  AE223FOBUQAUOSGVCDRABQGKGVSQ  2023-03-16 11:34:20.616   

           user_first_date  days_since_first  review_sequence  
0  2023-02-03 12:33:52.132                 0                1  
1  2023-04-28 08:53:34.935                 0                1  
2  2023-05-04 16:05:49.825                 0                1  
3  2023-05-12 07:14:26.657    

## Explore & Filter

In [7]:
# Filter by category
category_filter = 'Electronics'  # Change this to filter by category
df_filtered = df[df['category_name'] == category_filter]
print(f"Records in {category_filter}: {len(df_filtered):,}")
print(f"Users: {df_filtered['user_id'].nunique():,}")
print(f"Products: {df_filtered['parent_asin'].nunique():,}")
print(f"\nDate range: {df_filtered['date'].min()} to {df_filtered['date'].max()}")

Records in Electronics: 1,535,698
Users: 1,197,611
Products: 220,187

Date range: 2023-01-01 00:00:09.790 to 2023-06-29 23:57:29.655


In [8]:
# Find users with multiple reviews (retention candidates)
reviews_per_user = df.groupby('user_id').size()
multi_review_users = reviews_per_user[reviews_per_user >= 2]

print(f"\nUsers with 2+ reviews: {len(multi_review_users):,} ({len(multi_review_users)/df['user_id'].nunique()*100:.1f}%)")

# Sample a multi-review user to inspect
if len(multi_review_users) > 0:
    sample_user = multi_review_users.index[0]
    print(f"\nSample user with multiple reviews: {sample_user}")
    print(df[df['user_id'] == sample_user][['user_id', 'category_name', 'date', 'user_first_date', 'days_since_first', 'review_sequence', 'rating']].sort_values('date'))


Users with 2+ reviews: 375,415 (20.5%)

Sample user with multiple reviews: AE2226JU65H4HANS4GBCXVSNILPA
                             user_id category_name                     date  \
810362  AE2226JU65H4HANS4GBCXVSNILPA   Electronics  2023-03-17 09:53:17.505   
810363  AE2226JU65H4HANS4GBCXVSNILPA   Electronics  2023-03-17 10:12:20.813   

                user_first_date  days_since_first  review_sequence  rating  
810362  2023-03-17 09:53:17.505                 0                1     5.0  
810363  2023-03-17 09:53:17.505                 0                2     5.0  


## Edit Data (if needed)

In [9]:
# Example: Drop specific records
# df = df[df['rating'] >= 1]  # Keep all ratings >= 1

# Example: Filter to specific date range
# df = df[(df['date'] >= '2023-01-01') & (df['date'] <= '2023-06-30')]

# Example: Remove records with null user_id
# df = df[df['user_id'].notna()]

# After making edits, run this to see changes:
print(f"Current dataset: {len(df):,} records")
print(f"Users: {df['user_id'].nunique():,}")
print(f"Categories: {df['category_name'].nunique()}")

Current dataset: 2,516,345 records
Users: 1,827,354
Categories: 4


## Export Updated Data

In [ ]:
# Save updated data back to CSV (uncomment to use)
output_path = PROJECT_ROOT / 'data/cleaned/cleaned_reviews_edited.csv'
# df.to_csv(output_path, index=False)
# print(f"Saved to: {output_path}")
# print(f"Records: {len(df):,}")

# If happy with changes, replace original:
# import shutil
# shutil.copy(output_path, CLEANED_DATA_PATH)
# print(f"Replaced original: {CLEANED_DATA_PATH}")

print(f"Output path configured: {output_path}")
print("Uncomment df.to_csv() line above to save edits.")

## Validation Checks

In [ ]:
# Verify required specifications
print("=== VALIDATION CHECKLIST ===")

# Check 1: verified_purchase = True only
check1 = (df['verified_purchase'] == True).all()
print(f"✓ All records verified_purchase = True: {check1}")

# Check 2: Date range
from datetime import datetime
date_start = datetime(2023, 1, 1)
date_end = datetime(2023, 6, 30)
df['date_parsed'] = pd.to_datetime(df['date'])
in_range = ((df['date_parsed'] >= date_start) & (df['date_parsed'] <= date_end)).all()
print(f"✓ All records in range [2023-01-01, 2023-06-30]: {in_range}")

# Check 3: No duplicate (user_id, parent_asin, timestamp)
duplicates = df.duplicated(subset=['user_id', 'parent_asin', 'timestamp']).sum()
print(f"✓ Duplicate (user_id, parent_asin, timestamp) count: {duplicates}")

# Check 4: Categories
required_categories = {'Electronics', 'Video_Games', 'Software', 'Cell_Phones_and_Accessories'}
actual_categories = set(df['category_name'].unique())
print(f"✓ Required categories present: {required_categories.issubset(actual_categories)}")
print(f"  Categories in data: {actual_categories}")

# Check 5: No critical nulls
critical_fields = ['user_id', 'parent_asin', 'timestamp', 'verified_purchase', 'category_name']
nulls_in_critical = df[critical_fields].isnull().sum().sum()
print(f"✓ Null values in critical fields: {nulls_in_critical}")

=== VALIDATION CHECKLIST ===
✓ All records verified_purchase = True: True
✓ All records in range [2023-01-01, 2023-06-30]: True
✓ Duplicate (user_id, parent_asin, timestamp) count: 0
✓ Required categories present: True
  Categories in data: {'Video_Games', 'Electronics', 'Software', 'Cell_Phones_and_Accessories'}
✓ Null values in critical fields: 0


## Notes for Phase 2 (Graph Logic)

**Retention Definition (for reference):**
- A user is **retained** in a category if:
  - Within 90 days of their first review in that category
  - They post 2+ reviews on at least 2 distinct days

**Key columns for retention calculation:**
- `user_id`: Identify users
- `category_name`: Track category engagement
- `user_first_date`: First review date per user per category
- `days_since_first`: Days elapsed from first review
- `review_sequence`: Review number in order
- `date`: Exact timestamp for counting distinct days

**Data quality ready for graph logic**: ✓

In [ ]:
print(f"Number of unique products: {df['parent_asin'].nunique()}")
print(f"Number of unique users: {df['user_id'].nunique()}")

Number of unique products: 369157
Number of unique users: 1827354


In [11]:
df.sample(5)

,user_id,parent_asin,asin,timestamp,date,rating,verified_purchase,category_name,helpful_vote,user_first_date,days_since_first,review_sequence
1812863,AGNLEG27E4SA56NQ7GEUHGC2REOQ,B08Q2D3XR1,B00004T8R2,1679346890104,2023-03-20 17:14:50.104,5.0,True,Electronics,3,2023-03-20 17:14:50.104,0,1
991049,AEJ2I2Y34QSK3FUWE4AQFL5UOYQA,B0BZ57TZWL,B0931YVR2K,1677209362779,2023-02-23 22:29:22.779,5.0,True,Electronics,0,2023-02-23 22:29:22.779,0,1
869344,AE6VIT63PZP5I7HSVLYIFQG2RC7Q,B088ZGDKL3,B08913CR1L,1674351663904,2023-01-21 20:41:03.904,5.0,True,Electronics,0,2023-01-21 20:34:55.275,0,2
1838739,AGPQGZ3Y3HVR3KEIGYI4AW5JDAVQ,B0BYSK49CV,B0BDTPC3QG,1676036789876,2023-02-10 08:46:29.876,1.0,True,Electronics,0,2023-02-10 08:46:29.876,0,1
1163854,AEXIY55LXZXJMSMYXQZYPKUIJGRQ,B0BVPT6QJ4,B0BRH9718D,1678677209139,2023-03-12 23:13:29.139,5.0,True,Electronics,0,2023-02-26 17:40:23.423,14,7


In [ ]:
df.columns

Index(['user_id', 'parent_asin', 'asin', 'timestamp', 'date', 'rating',
       'verified_purchase', 'category_name', 'helpful_vote', 'user_first_date',
       'days_since_first', 'review_sequence', 'date_parsed'],
      dtype='object')

In [19]:
df[df['user_id'] == "AGWMT2QORZIH3ITWUO5CRPK2OH5A"]

,user_id,parent_asin,asin,timestamp,date,rating,verified_purchase,category_name,helpful_vote,user_first_date,days_since_first,review_sequence
585812,AGWMT2QORZIH3ITWUO5CRPK2OH5A,B08C5GNDWF,B00P936188,1680623215909,2023-04-04 11:46:55.909,4.0,True,Cell_Phones_and_Accessories,2,2023-04-04 11:46:55.909,0,1
585813,AGWMT2QORZIH3ITWUO5CRPK2OH5A,B0964J2V9V,B0964HKSRB,1680913518846,2023-04-07 20:25:18.846,5.0,True,Cell_Phones_and_Accessories,0,2023-04-04 11:46:55.909,3,2
585814,AGWMT2QORZIH3ITWUO5CRPK2OH5A,B08RSN1VR2,B08RSNRV6Z,1680918470787,2023-04-07 21:47:50.787,4.0,True,Cell_Phones_and_Accessories,0,2023-04-04 11:46:55.909,3,3
585815,AGWMT2QORZIH3ITWUO5CRPK2OH5A,B09SP82PQ4,B09KC2XB6W,1681077790334,2023-04-09 18:03:10.334,4.0,True,Cell_Phones_and_Accessories,1,2023-04-04 11:46:55.909,5,4
585816,AGWMT2QORZIH3ITWUO5CRPK2OH5A,B01BGG8LHK,B01AUVS3HU,1681367170708,2023-04-13 02:26:10.708,4.0,True,Cell_Phones_and_Accessories,1,2023-04-04 11:46:55.909,8,5
...,...,...,...,...,...,...,...,...,...,...,...,...
2476955,AGWMT2QORZIH3ITWUO5CRPK2OH5A,B09R4K5ZTZ,B077ZGRQM2,1685176114035,2023-05-27 04:28:34.035,5.0,True,Video_Games,1,2023-04-04 11:44:02.955,52,214
2476956,AGWMT2QORZIH3ITWUO5CRPK2OH5A,B079C7QGLQ,B071WPKD5P,1685181345520,2023-05-27 05:55:45.520,5.0,True,Video_Games,0,2023-04-04 11:44:02.955,52,215
2476957,AGWMT2QORZIH3ITWUO5CRPK2OH5A,B07R4C3ZVH,B078Y4FR14,1685183579866,2023-05-27 06:32:59.866,5.0,True,Video_Games,0,2023-04-04 11:44:02.955,52,216
2476958,AGWMT2QORZIH3ITWUO5CRPK2OH5A,B072MQNKYV,B071G5HZ7F,1685303705990,2023-05-28 15:55:05.990,5.0,True,Video_Games,0,2023-04-04 11:44:02.955,54,217


In [30]:
user_review_freq_dist = pd.DataFrame(df.user_id.value_counts()).reset_index()
user_review_freq_dist.head(20)

,user_id,count
0,AGWMT2QORZIH3ITWUO5CRPK2OH5A,285
1,AGWBKZXJM7FNL3MXA2AQYE2S444A,128
2,AE24PGKOXDPI2NDEKVRB2GGWUNOA,117
3,AFTZWAK3ZHAPCNSOT5GCKQDECBTQ,115
4,AGH5ZCKJNRAZ7L7TOHMV35GMF2SQ,108
5,AGFN3252BBTYUJUUDQMAGYZNUT5A_1,103
6,AGP5T7EZG6UWWQE6QZOG5NSJULRQ,78
7,AEW6RDE22OHQGQGYKJH7E4T747HA,76
8,AEAVWVRGL5XNUQMS2KWQM5D6ZO4Q,72
9,AE7334SMJRQHADELSR6MHVOGCCIA,72


In [44]:
df.date.max()

'2023-06-29 23:59:46.366'